## Project goal

Get familiar with small GPT training.  
Very strongly inspired by Andrej Karpathy's [youtube video on nanoGPT](https://youtu.be/kCc8FmEb1nY).

Outline:
- load data
- tokenize
- encode characters as integers
- get minibatches of sequences
- first model (bigram) $P(next|last)$
- token embeddings and position embeddings
- self-attention
- multi-head attention
- blocks of multi-head attention
- adding reasoning layers, skip connections, layer normalization

## Load data

In [ ]:
filename = 'tinyshakespeare.txt' # 'tinymoliere.txt'
with open(filename, 'r', encoding='utf-8') as f:
    text = f.read()

In [ ]:
print("number of characters:", len(text))

In [ ]:
print(text[:500])

## Tokenize

Our token are just characters.

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("number of tokens (codebook size):", vocab_size)
print(chars)

Token encoding = token number in the 'chars' sequence.

In [ ]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encode a string as a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decode a list of integers as a string

In [ ]:
print(encode('hello world'))
print(decode(encode('hello world')))

Alternative: use tiktoken (OpenAI's tokenization library, based on BPE).

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("gpt2")
assert enc.decode(enc.encode('hello world')) == 'hello world'

In [ ]:
print("vocabulary length (codebook size):",enc.n_vocab)
print(enc.encode('hello world'))

For now, we'll keep the character-level encoding to keep things simple.

## Encode

In [ ]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:500])

## Split dataset

In [ ]:
n = int(.9*len(data))
train_data = data[:n]
val_data = data[n:]

## Extract a chunk from the data

In [ ]:
block_size = 8
offset = 0
train_data[offset:offset+block_size+1]

This holds 'block_size+1' training examples:
- predict 2nd token based on 1st
- predict 3rd token based on 1st and 2nd
- etc. until predict 'block_size+1' token based on 'block_size' previous ones

Actually, this could also hold even more examples: 'predict 4th token based on 3rd only' works as well. But that would yield an imbalance in the distribution of training context lengths: with the above we have a uniform distribution across context lengths, whereas if we were listing all possible subsequences we would have a large imbalance towards short contexts.

Let's display these 'block_size+1' examples:

In [ ]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"predict {target} based on {context}")

## Draw a minibatch

In [ ]:
torch.manual_seed(1337)
batch_size = 4 # number of sequences in a minibatch
block_size = 8 # context length

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y

In [ ]:
xb, yb = get_batch('train')
print('inputs shape:', xb.shape)
print(xb)
print('targets shape:', yb.shape)
print(yb)

In [ ]:
for b in range(batch_size):
    for t in range(block_size):
        context = xb[b,:t+1]
        target = yb[b,t]
        print(f"predict {target} based on {context}")

## The bigram language model

In the bigram language model, the distribution on the next token only depends on the current token. So we have to store a `vocab_size` vector of values for each token, and we have to do that for all tokens. Thus we have to store a `vocab_size`x`vocab_size` matrix.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # The logits for the next token are simply stored in a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (B,T)-shaped tensors of integers
        logits = self.token_embedding_table(idx)
        # at this point, logits are a (B,T,C)-shaped tensor
        # Pytorch expects a (n_data,n_classes) tensor to compute a CE loss
        # so we reshape the logits if we need to compute a loss
        # otherwise, keep the logits in the shape they are
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

In [ ]:
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(xb.shape)
print(yb.shape)
print(logits.shape)
print(loss)

Vocabulary holds 65 tokens and the model is untrained to should be close to a uniform distribution. So the loss should be close to -ln(1/65):

In [ ]:
import numpy as np
print(np.log(vocab_size))

Let's add a method that does token generation for `max_new_token` tokens and appends them to the input.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # The logits for the next token are simply stored in a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (B,T)-shaped tensors of integers
        logits = self.token_embedding_table(idx)
        # at this point, logits are a (B,T,C)-shaped tensor
        # Pytorch expects a (n_data,n_classes) tensor to compute a CE loss
        # so we reshape the logits if we need to compute a loss
        # otherwise, keep the logits in the shape they are
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is a (B,T) array of indices
        for _ in range(max_new_tokens):
            # get prediction for last column of idx
            logits, loss = self(idx)
            logits = logits[:,-1,:]
            # apply softmax
            probs = F.softmax(logits, dim=-1)
            # sample next token
            idx_next = torch.multinomial(probs, num_samples=1)
            # append to sequence
            idx = torch.cat((idx,idx_next), dim=1)
        return idx

Let's generate text starting from token 0.  
So we create a batch with a single sequence, containing a single element, which is the token 0.  
Let's pass it to our 'decode' function, then generate the 100 next tokens.

In [ ]:
# first token
idx = torch.zeros((1,1), dtype=torch.long)
decode(idx[0].tolist())

In [ ]:
m = BigramLanguageModel(vocab_size)
new_seq = m.generate(idx, max_new_tokens=100)
print("predicted sequence:\n", new_seq)
print("... as a string of characters:\n", decode(new_seq[0].tolist()))

## Training the bigram model

In [ ]:
optimizer = torch.optim.Adam(m.parameters(), lr=1e-3) # usual value of 3e-4 is not necessary here as our model is quite small

In [ ]:
batch_size = 32
for steps in range(10000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
print(loss.item())

Let's try to make a prediction.

In [ ]:
new_seq = m.generate(idx, max_new_tokens=400)
print("predicted sequence:\n", new_seq)
print("... as a string of characters:\n", decode(new_seq[0].tolist()))

## Wrapping up on data loading and bigram model training

Let's encapsulate all this in a single script.

In [ ]:
# %load bigram.py

This could be better but it's just a bigram model: only the last character conditions the prediction. So, at this stage, we'd like the model to become a bit more clever and start looking at all tokens in the sequence (not just the last) to make a prediction.

## Introducing token embeddings and position embeddings in the bigram model code

So far, tokens directly map towards the logits used to predict the next token. Let's introduce an abstraction level where each token maps to an embedding of size `n_embd`, which can then be decoded into logits.

- remove `vocab_size` from constructor since it's a global variable
- introduce `n_embd=32` the token embedding dimension as a global variable
- `token_embedding_table` shape becomes (vocab_size,n_embd)
- calling `token_embedding_table(idx)` now returns the embedding of the token, not the logits for the next token (rename the output from `logits` to `tok_emb`)
- introduce a `lm_head` linear layer that turns an embedding into logits

Introducing this embedding level serves no purpose for now, since the embedding is a lookup table and the lm_head is a linear operation. So, instead of storing `vocab_size=65` probability values for each token line, we now store `n_embd=32` values, and then also store a matrix `lm_head` of shape `(n_embd, vocab_size)` that maps these values back to probability values. This is just a way to introduce token embeddings.

Also, token embeddings are position invariant. The position of a token in the sequence might have an importance for next token prediction, so let's add a position embedding of size `n_embd` for each token.

- introduce a `position_embedding_table` embedding that maps positions in the sequence to embeddings
- redefine the forward pass to sum the token embedding with the position embedding and pass the result through `lm_head`

Warning: this causes an indexing problem in the forward pass, as the sequence length can grow larger than `block_size` while the `position_embedding_table` cannot accomodate positions larger than `block_size`. To prevent this, in `generate`, only call `forward` on the last `block_size` elements.

In [ ]:
# %load bigram_v2.py

## Personal note

We want to go from $P(s_{t+1}|s_{t})$ to $P(s_{t+1}|s_0,\ldots,s_t)$.  
Gaussian Processes learned that by making the Gaussian assumption and learning the kernel parameters.  

## The math trick in self-attention

Currently, decision for the next token relies only on the last one.  
Let's make this dependent on the *average* of all previous ones in the `block_size` sequence. So let's start by computing this average.

In [ ]:
import torch

torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time (context length), channels (codebook size)
x = torch.randn(B,T,C)
x.shape

In [ ]:
# we want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C)) # "bow" is for Bag of Words
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # this has shape (t,C)
        xbow[b,t] = torch.mean(xprev, 0)

In [ ]:
x[0]

In [ ]:
xbow[0]

Let's do the same thing but with matrix multiplication, to take advantage of built-in parallelisation.

In [ ]:
# Version 2: matrix multiplication
# this had complexity BxT
# let's do that more efficiently with matrix multiplication to take advantage of parallelization
wei = torch.tril(torch.ones(T,T)) # torch.tril takes the lower triangular values
wei = wei / wei.sum(1, keepdim=True) # normalize each row
xbow2 = wei @ x 
# when torch sees a matmul (T,T) @ (B,T,C), it morphs it into (B,T,T) @ (B,T,C) 
# and does the matmul element-wise on the first dimension and stacks them. 
# So it does B times a (T,T) @ (T,C) operation.
# So we get a (B,T,C) tensor at the end
print(torch.allclose(xbow,xbow2)) # might fail because of precision issues
print(torch.max(xbow-xbow2))

In [ ]:
xbow[0], xbow2[0]

Eventually, we will not perform a uniform average but rather a weighted one  and we will learn the weights. In fact, to make sure to weights are always positive, we will learn the logits and apply a softmax. So let's do the same thing one last time with a softmax function.

In [ ]:
# Version 3: Softmax
from torch.nn import functional as F
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf')) # replace 0 values by -inf, in wei, where tril==0
print(wei)
wei = F.softmax(wei, dim=-1) # get the same weight matrix as before
xbow3 = wei @ x
torch.allclose(xbow,xbow3)
print(torch.max(xbow-xbow3))

The printed version of `wei` above can be interpreted as *affinities*.  
Actual weights in the matrix that performs the averaging operation are the softmax over affinities.  
When we predict the nth token, only the previous ones participate and have affinities with the token, not the ones in the future.

## Self-attention

So far, the weights attending to previous positions were uniform (the affinities were all zero). We want these weights to be data dependent.  

For instance, token number 3 might be asking the question "which other tokens should I have affinity with?". This is a *query*. This *query* is directed at all other tokens.  
The query is a vector that stands for "what am I looking for?".

All other tokens will have a *key*. The key is a vector that encodes "what do I contain?" or "what do I mean?".

If the *key* matches the *query*, then the affinity should be high.  

So each token should emit 2 embedding vectors: one query $q(x)$ and one key $k(x)$, based on its embedding $x$. The attention score between a token and any other one is a function of the pair $(q(x),k(x))$.

In dot-product self-attention, queries and keys lie in the same vector space and the affinity is $q(x)^Tk(x)$. The attention score is the softmax of affinities.

Recall that the embedding $x$, as we have defined it so far, holds the information of "which word am I in the codebook?" and the information of "what position am I in?".

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)
B,T,C = 4,8,32 # batch size, context window, channels
x = torch.randn(B,T,C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
k = key(x) # (B,T,16)
q = query(x) # (B,T,16)

wei = q @ k.transpose(-2,-1) # (B,T,16) @ (B,16,T) ---> (B,T,T)
# when doing the transpose operation, we want to keep the batch dimension untouched

# we still want to mask future tokens in the attention mechanism
tril = torch.tril(torch.ones(T,T)) 
wei = wei.masked_fill(tril==0, float('-inf'))
wei = F.softmax(wei,dim=-1)
out = wei @ x

print(out.shape)

In [ ]:
wei[0]

In the code above, the prediction for a given token is the average of all its predecessors' embeddings, with weights given by the attention mechanism.  

Let's add one more level of flexibility: we want the prediction for a token to be the average of all its predecessors' *values* $v(x)$ instead of their embeddings $x$.  
Such *values* are "what I will communicate in the answer to the query".

Let's make the value live in a `head_size` dimensional space too.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)
B,T,C = 4,8,32 # batch size, context window, channels
x = torch.randn(B,T,C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x) # (B,T,16)
q = query(x) # (B,T,16)
v = value(x) # (B,T,16)

wei = q @ k.transpose(-2,-1) # (B,T,16) @ (B,16,T) ---> (B,T,T)
# when doing the transpose operation, we want to keep the batch dimension untouched

# we still want to mask future tokens in the attention mechanism
tril = torch.tril(torch.ones(T,T)) 
wei = wei.masked_fill(tril==0, float('-inf'))
wei = F.softmax(wei,dim=-1)
out = wei @ v

print(out.shape)

Notes:
- Attention is a data-dependent communication mechanism across items. It could be applied over other data than time series.
- Positional embedding break the position invariance property to enrich representation (could be regained by removing positional encodings).
- cross-attention refers to the case where the query comes from an element $x$ and the keys and values come from a separate collection of items $y$ living in another space, that we want to pool information from. So we would have $q(x),k(y),v(y)$ elements in the attention mechanism. Self-attention is when $x$ and $y$ live in the same space.
- The operation of computing all $q(x)$, $k(x)$, and $v(x)$, and outputing the next token prediction is called an attention head.
- Scaled dot-product attention:
$$Attention(Q,K,V) = softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$
The scaling is there to ensure unit-variance in the $qk^T$ for unit variance in the inputs $q$ and $k$, as shown in the example below. This is important to keep the inputs of the softmax in a reasonable range, so that its outputs are not too close to zero or one.

In [ ]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2,-1)

print(k.var())
print(q.var())
print(wei.var())
print(head_size)

wei = wei * head_size**-.5

print(wei.var())

## Single head attention-based language model

In [ ]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size,block_size)))

    def forward(self,x):
        B,T,C = x.shape
        k = self.key(x)   # B,T,C warning, here C stands for head_size
        q = self.query(x) # B,T,C
        # compute affinities and attention scores
        wei = q @ k.transpose(-2,-1) * C**-.5 # (B,T,T)
        wei = wei.masked_fill(self.tril[:T,T:]==0, float('-inf')) # (B,T,T)
        wei = F.softmax(wei, dim=-1) # (B,T,T)
        # aggregate values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B,T,C)
        return out

Let's use this attention head in our previous code to make the next-token prediction depend on the output of the attention mechanism. Use `head_size`=`n_embd`.

In [ ]:
# Single attention head language model
class SingleAttentionHeadLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        # The logits for the next token are simply stored in a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.sa_head = Head(n_embd) # sa_head stands for self-attention head
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T  = idx.shape
        # idx and targets are both (B,T)-shaped tensors of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C) 
        # warning here, T might grow larger than block_size if we generate more tokens than block_size, which will cause an index error in the previous line, but we can ignore that for now.
        x = tok_emb + pos_emb # the sum broadcasts the (T,C) tensor across batches
        x = self.sa_head(x)
        logits = self.lm_head(x)
        # at this point, logits are a (B,T,C)-shaped tensor
        # Pytorch expects a (n_data,n_classes) tensor to compute a CE loss
        # so we reshape the logits if we need to compute a loss
        # otherwise, keep the logits in the shape they are
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is a (B,T) array of indices
        for _ in range(max_new_tokens):
            # get prediction for last column of idx
            idx_cond = idx[:,-block_size:] # crop idx to the last block_size tokens
            logits, loss = self(idx_cond)
            logits = logits[:,-1,:]
            # apply softmax
            probs = F.softmax(logits, dim=-1)
            # sample next token
            idx_next = torch.multinomial(probs, num_samples=1)
            # append to sequence
            idx = torch.cat((idx,idx_next), dim=1)
        return idx

The script below trains a the model above.

The only changes to the bigram model hyperparameters are:
- learning rate decreased to $1e-3$
- `max_iters` increased to 5000
- `eval_interval` increased to 500

In [ ]:
# %load single_attention_head.py
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 32
block_size = 8
max_iters = 5000
eval_interval = 500
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 32

# seeding (for reproducibility)
torch.manual_seed(1337)

# open text file
filename = 'tinyshakespeare.txt' # 'tinymoliere.txt'
with open(filename, 'r', encoding='utf-8') as f:
    text = f.read()

# tokens
chars = sorted(list(set(text)))
vocab_size = len(chars)

# encode and decode strings
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encode a string as a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decode a list of integers as a string

# split dataset
data = torch.tensor(encode(text), dtype=torch.long)
n = int(.9*len(data))
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x,y

# estimate training and validation losses (for monitoring purposes)
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# single attention head
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size,block_size)))

    def forward(self,x):
        B,T,C = x.shape
        k = self.key(x)   # B,T,C warning, here C stands for head_size
        q = self.query(x) # B,T,C
        # compute affinities and attention scores
        wei = q @ k.transpose(-2,-1) * C**-.5 # (B,T,T)
        wei = wei.masked_fill(self.tril[:T,:T]==0, float('-inf')) # (B,T,T)
        wei = F.softmax(wei, dim=-1) # (B,T,T)
        # aggregate values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B,T,C)
        return out

# Single attention head language model
class SingleAttentionHeadLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        # The logits for the next token are simply stored in a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.sa_head = Head(n_embd) # sa_head stands for self-attention head
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T  = idx.shape
        # idx and targets are both (B,T)-shaped tensors of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C) 
        # warning here, T might grow larger than block_size if we generate more tokens than block_size, which will cause an index error in the previous line, but we can ignore that for now.
        x = tok_emb + pos_emb # the sum broadcasts the (T,C) tensor across batches
        x = self.sa_head(x)
        logits = self.lm_head(x)
        # at this point, logits are a (B,T,C)-shaped tensor
        # Pytorch expects a (n_data,n_classes) tensor to compute a CE loss
        # so we reshape the logits if we need to compute a loss
        # otherwise, keep the logits in the shape they are
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is a (B,T) array of indices
        for _ in range(max_new_tokens):
            # get prediction
            idx_cond = idx[:,-block_size:] # crop idx to the last block_size tokens
            logits, loss = self(idx_cond)
            logits = logits[:,-1,:]
            # apply softmax
            probs = F.softmax(logits, dim=-1)
            # sample next token
            idx_next = torch.multinomial(probs, num_samples=1)
            # append to sequence
            idx = torch.cat((idx,idx_next), dim=1)
        return idx

model = SingleAttentionHeadLanguageModel()
m = model.to(device)

# optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
for iter in range(max_iters):
    # periodically evaluate the validation loss
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    # sample minibatch
    xb, yb = get_batch('train')
    # evaluate loss and take gradient step
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
# generate from model
idx = torch.zeros((1,1), dtype=torch.long, device=device)
new_seq = m.generate(idx, max_new_tokens=500)
print("predicted sequence:\n", new_seq)
print("... as a string of characters:\n", decode(new_seq[0].tolist()))


## Multi-head attention

Instead of a single head, let's use several and just concatenate the output.  
The key idea is to let one head look for a certain type of information (a certain type of query / key / value), while another searches for another type of info, and so on.  
For instance, the first head could look for nearby vowels, the second could look for the presence of the past tense, etc. And then all these independent predictions form the output of the attention layer.

In [ ]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

    def forward(self,x):
        return torch.cat([h(x) for h in self.heads], dim=-1)

To keep the output dimension constant, use 4 heads, each with an embedding dimension of n_embd/4.

In [ ]:
# %load multi_head_attention.py
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 32
block_size = 8
max_iters = 5000
eval_interval = 500
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 32

# seeding (for reproducibility)
torch.manual_seed(1337)

# open text file
filename = 'tinyshakespeare.txt' # 'tinymoliere.txt'
with open(filename, 'r', encoding='utf-8') as f:
    text = f.read()

# tokens
chars = sorted(list(set(text)))
vocab_size = len(chars)

# encode and decode strings
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encode a string as a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decode a list of integers as a string

# split dataset
data = torch.tensor(encode(text), dtype=torch.long)
n = int(.9*len(data))
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x,y

# estimate training and validation losses (for monitoring purposes)
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# single attention head
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size,block_size)))

    def forward(self,x):
        B,T,C = x.shape
        k = self.key(x)   # B,T,C warning, here C stands for head_size
        q = self.query(x) # B,T,C
        # compute affinities and attention scores
        wei = q @ k.transpose(-2,-1) * C**-.5 # (B,T,T)
        wei = wei.masked_fill(self.tril[:T,:T]==0, float('-inf')) # (B,T,T)
        wei = F.softmax(wei, dim=-1) # (B,T,T)
        # aggregate values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B,T,C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

    def forward(self,x):
        return torch.cat([h(x) for h in self.heads], dim=-1)

# Single attention head language model
class MultiHeadAttentionLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        # The logits for the next token are simply stored in a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.sa_heads = MultiHeadAttention(num_heads=4, head_size=n_embd//4) # 4 heads, each with size n_embd//4
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T  = idx.shape
        # idx and targets are both (B,T)-shaped tensors of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C) 
        # warning here, T might grow larger than block_size if we generate more tokens than block_size, which will cause an index error in the previous line, but we can ignore that for now.
        x = tok_emb + pos_emb # the sum broadcasts the (T,C) tensor across batches
        x = self.sa_heads(x)
        logits = self.lm_head(x)
        # at this point, logits are a (B,T,C)-shaped tensor
        # Pytorch expects a (n_data,n_classes) tensor to compute a CE loss
        # so we reshape the logits if we need to compute a loss
        # otherwise, keep the logits in the shape they are
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is a (B,T) array of indices
        for _ in range(max_new_tokens):
            # get prediction
            idx_cond = idx[:,-block_size:] # crop idx to the last block_size tokens
            logits, loss = self(idx_cond)
            logits = logits[:,-1,:]
            # apply softmax
            probs = F.softmax(logits, dim=-1)
            # sample next token
            idx_next = torch.multinomial(probs, num_samples=1)
            # append to sequence
            idx = torch.cat((idx,idx_next), dim=1)
        return idx

model = MultiHeadAttentionLanguageModel()
m = model.to(device)

# optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
for iter in range(max_iters):
    # periodically evaluate the validation loss
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    # sample minibatch
    xb, yb = get_batch('train')
    # evaluate loss and take gradient step
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
# generate from model
idx = torch.zeros((1,1), dtype=torch.long, device=device)
new_seq = m.generate(idx, max_new_tokens=500)
print("predicted sequence:\n", new_seq)
print("... as a string of characters:\n", decode(new_seq[0].tolist()))


## Adding a feedforward layer after the attention output

Currently, the output of all attention heads is just concatenated and fed to the (linear) language model head to compute the logits. Let's add an inbetween feedforward ReLU layer so that there can be some "reasoning" on the output of the attention heads.

In [ ]:
class FeedForward(nn.Module):
    """ ReLU layer """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, n_embd),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
# %load multi_head_attention_v2.py
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 32
block_size = 8
max_iters = 5000
eval_interval = 500
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 32

# seeding (for reproducibility)
torch.manual_seed(1337)

# open text file
filename = 'tinyshakespeare.txt' # 'tinymoliere.txt'
with open(filename, 'r', encoding='utf-8') as f:
    text = f.read()

# tokens
chars = sorted(list(set(text)))
vocab_size = len(chars)

# encode and decode strings
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encode a string as a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decode a list of integers as a string

# split dataset
data = torch.tensor(encode(text), dtype=torch.long)
n = int(.9*len(data))
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x,y

# estimate training and validation losses (for monitoring purposes)
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# single attention head
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size,block_size)))

    def forward(self,x):
        B,T,C = x.shape
        k = self.key(x)   # B,T,C warning, here C stands for head_size
        q = self.query(x) # B,T,C
        # compute affinities and attention scores
        wei = q @ k.transpose(-2,-1) * C**-.5 # (B,T,T)
        wei = wei.masked_fill(self.tril[:T,:T]==0, float('-inf')) # (B,T,T)
        wei = F.softmax(wei, dim=-1) # (B,T,T)
        # aggregate values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B,T,C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

    def forward(self,x):
        return torch.cat([h(x) for h in self.heads], dim=-1)

class FeedForward(nn.Module):
    """ ReLU layer """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, n_embd),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)

# Single attention head language model
class MultiHeadAttentionLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        # The logits for the next token are simply stored in a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.sa_heads = MultiHeadAttention(num_heads=4, head_size=n_embd//4) # 4 heads, each with size n_embd//4
        self.ffwd = FeedForward(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T  = idx.shape
        # idx and targets are both (B,T)-shaped tensors of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C) 
        # warning here, T might grow larger than block_size if we generate more tokens than block_size, which will cause an index error in the previous line, but we can ignore that for now.
        x = tok_emb + pos_emb # the sum broadcasts the (T,C) tensor across batches
        x = self.sa_heads(x)
        x = self.ffwd(x)
        logits = self.lm_head(x)
        # at this point, logits are a (B,T,C)-shaped tensor
        # Pytorch expects a (n_data,n_classes) tensor to compute a CE loss
        # so we reshape the logits if we need to compute a loss
        # otherwise, keep the logits in the shape they are
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is a (B,T) array of indices
        for _ in range(max_new_tokens):
            # get prediction
            idx_cond = idx[:,-block_size:] # crop idx to the last block_size tokens
            logits, loss = self(idx_cond)
            logits = logits[:,-1,:]
            # apply softmax
            probs = F.softmax(logits, dim=-1)
            # sample next token
            idx_next = torch.multinomial(probs, num_samples=1)
            # append to sequence
            idx = torch.cat((idx,idx_next), dim=1)
        return idx

model = MultiHeadAttentionLanguageModel()
m = model.to(device)

# optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
for iter in range(max_iters):
    # periodically evaluate the validation loss
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    # sample minibatch
    xb, yb = get_batch('train')
    # evaluate loss and take gradient step
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
# generate from model
idx = torch.zeros((1,1), dtype=torch.long, device=device)
new_seq = m.generate(idx, max_new_tokens=500)
print("predicted sequence:\n", new_seq)
print("... as a string of characters:\n", decode(new_seq[0].tolist()))


## Blocks of multihead attention and reasoning, with skip connections

In [ ]:
class Block(nn.Module):
    """ Transformer block: communication then computation """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)

    def forward(self, x):
        x = self.sa(x)
        x = self.ffwd(x)
        return x

Let's build a network with a feedforward sequence of 3 such blocks.

The network is starting to become quite deep and optimization would start to struggle. So let's also add skip connections to facilitate gradient flow.

In [ ]:
class Block(nn.Module):
    """ Transformer block: communication then computation """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)

    def forward(self, x):
        x = x + self.sa(x) # skip connection
        x = x + self.ffwd(x) # skip connection
        return x

Since we have added the skip connections, we are re-injecting the result of computation (`sa(x)` and `ffwd(x)`) into the "residual pathway" (`x`). So we need to introduce an degree of freedom to project these results onto that pathway: we need to add a linear layer at the end of the `MultiHeadAttention` and `Feedforward` classes.

In [ ]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self,x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.prof(out)

In [ ]:
class FeedForward(nn.Module):
    """ ReLU layer """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, n_embd),
            nn.ReLU(),
            nn.Linear(n_embd, n_embd),
        )

    def forward(self, x):
        return self.net(x)

One last improvement coming from the *Attention is all you need* paper: the inner layer in the `FeedForward` module has dimensionality `4*n_embd`.

In [ ]:
# %load multi_head_attention_v3.py
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 32
block_size = 8
max_iters = 5000
eval_interval = 500
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 32

# seeding (for reproducibility)
torch.manual_seed(1337)

# open text file
filename = 'tinyshakespeare.txt' # 'tinymoliere.txt'
with open(filename, 'r', encoding='utf-8') as f:
    text = f.read()

# tokens
chars = sorted(list(set(text)))
vocab_size = len(chars)

# encode and decode strings
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encode a string as a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decode a list of integers as a string

# split dataset
data = torch.tensor(encode(text), dtype=torch.long)
n = int(.9*len(data))
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x,y

# estimate training and validation losses (for monitoring purposes)
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# single attention head
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size,block_size)))

    def forward(self,x):
        B,T,C = x.shape
        k = self.key(x)   # B,T,C warning, here C stands for head_size
        q = self.query(x) # B,T,C
        # compute affinities and attention scores
        wei = q @ k.transpose(-2,-1) * C**-.5 # (B,T,T)
        wei = wei.masked_fill(self.tril[:T,:T]==0, float('-inf')) # (B,T,T)
        wei = F.softmax(wei, dim=-1) # (B,T,T)
        # aggregate values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B,T,C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self,x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.proj(out)

class FeedForward(nn.Module):
    """ ReLU layer """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication then computation """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)

    def forward(self, x):
        x = x + self.sa(x) # skip connection
        x = x + self.ffwd(x) # skip connection
        return x

# Single attention head language model
class MultiHeadAttentionLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        # The logits for the next token are simply stored in a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(
            Block(n_embd, n_head=4),
            Block(n_embd, n_head=4),
            Block(n_embd, n_head=4),
        )
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T  = idx.shape
        # idx and targets are both (B,T)-shaped tensors of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C) 
        # warning here, T might grow larger than block_size if we generate more tokens than block_size, which will cause an index error in the previous line, but we can ignore that for now.
        x = tok_emb + pos_emb # the sum broadcasts the (T,C) tensor across batches
        x = self.blocks(x)
        logits = self.lm_head(x)
        # at this point, logits are a (B,T,C)-shaped tensor
        # Pytorch expects a (n_data,n_classes) tensor to compute a CE loss
        # so we reshape the logits if we need to compute a loss
        # otherwise, keep the logits in the shape they are
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is a (B,T) array of indices
        for _ in range(max_new_tokens):
            # get prediction
            idx_cond = idx[:,-block_size:] # crop idx to the last block_size tokens
            logits, loss = self(idx_cond)
            logits = logits[:,-1,:]
            # apply softmax
            probs = F.softmax(logits, dim=-1)
            # sample next token
            idx_next = torch.multinomial(probs, num_samples=1)
            # append to sequence
            idx = torch.cat((idx,idx_next), dim=1)
        return idx

model = MultiHeadAttentionLanguageModel()
m = model.to(device)

# optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
for iter in range(max_iters):
    # periodically evaluate the validation loss
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    # sample minibatch
    xb, yb = get_batch('train')
    # evaluate loss and take gradient step
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
# generate from model
idx = torch.zeros((1,1), dtype=torch.long, device=device)
new_seq = m.generate(idx, max_new_tokens=500)
print("predicted sequence:\n", new_seq)
print("... as a string of characters:\n", decode(new_seq[0].tolist()))


## Layer normalization

Let's make the optimization more efficient by implementing layer normalization.

In [ ]:
class LayerNorm:
    def __init__(self, dim, eps=1e-5):
        self.eps = eps
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

    def __call__(self, x):
        xmean = x.mean(1, keepdim=True) # Layer-wise mean
        xvar = x.var(1, keepdim=True) # Layer-wise variance
        xhat = (x-xmean)/torch.sqrt(xvar + self.eps) # normalize to unit variance
        out = self.gamma * xhat + self.beta
        return out

    def parameters(self):
        return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm(100)
x = torch.randn(32, 100) # batch of 32 100-dimensional vectors
x = module(x)

print(x.shape)
print(x[:,0].mean(), x[:,0].std())
print(x[0,:].mean(), x[0,:].std())

Slight deviation from the original *Attention is all you need paper*: recent years have shown that implementing layer normalization *before* multi-head attention and feedforward modules was beneficial compared to putting them after. So that's what we will do.

Also, we'll use pytorch built-in `LayerNorm` layer.

In [ ]:
class Block(nn.Module):
    """ Transformer block: communication then computation """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x)) # skip connection
        x = x + self.ffwd(self.ln2(x)) # skip connection
        return x

Also add layer normalization after the 3 `Block`, so we feed a clean, normalized vector into the final logits prediction output layer.

In [1]:
# %load multi_head_attention_v4.py
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 32
block_size = 8
max_iters = 5000
eval_interval = 500
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 32

# seeding (for reproducibility)
torch.manual_seed(1337)

# open text file
filename = 'tinyshakespeare.txt' # 'tinymoliere.txt'
with open(filename, 'r', encoding='utf-8') as f:
    text = f.read()

# tokens
chars = sorted(list(set(text)))
vocab_size = len(chars)

# encode and decode strings
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encode a string as a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decode a list of integers as a string

# split dataset
data = torch.tensor(encode(text), dtype=torch.long)
n = int(.9*len(data))
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x,y

# estimate training and validation losses (for monitoring purposes)
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# single attention head
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size,block_size)))

    def forward(self,x):
        B,T,C = x.shape
        k = self.key(x)   # B,T,C warning, here C stands for head_size
        q = self.query(x) # B,T,C
        # compute affinities and attention scores
        wei = q @ k.transpose(-2,-1) * C**-.5 # (B,T,T)
        wei = wei.masked_fill(self.tril[:T,:T]==0, float('-inf')) # (B,T,T)
        wei = F.softmax(wei, dim=-1) # (B,T,T)
        # aggregate values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B,T,C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self,x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.proj(out)

class FeedForward(nn.Module):
    """ ReLU layer """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication then computation """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x)) # skip connection
        x = x + self.ffwd(self.ln2(x)) # skip connection
        return x

# Single attention head language model
class MultiHeadAttentionLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        # The logits for the next token are simply stored in a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(
            Block(n_embd, n_head=4),
            Block(n_embd, n_head=4),
            Block(n_embd, n_head=4),
            nn.LayerNorm(n_embd),
        )
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T  = idx.shape
        # idx and targets are both (B,T)-shaped tensors of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C) 
        # warning here, T might grow larger than block_size if we generate more tokens than block_size, which will cause an index error in the previous line, but we can ignore that for now.
        x = tok_emb + pos_emb # the sum broadcasts the (T,C) tensor across batches
        x = self.blocks(x)
        logits = self.lm_head(x)
        # at this point, logits are a (B,T,C)-shaped tensor
        # Pytorch expects a (n_data,n_classes) tensor to compute a CE loss
        # so we reshape the logits if we need to compute a loss
        # otherwise, keep the logits in the shape they are
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is a (B,T) array of indices
        for _ in range(max_new_tokens):
            # get prediction
            idx_cond = idx[:,-block_size:] # crop idx to the last block_size tokens
            logits, loss = self(idx_cond)
            logits = logits[:,-1,:]
            # apply softmax
            probs = F.softmax(logits, dim=-1)
            # sample next token
            idx_next = torch.multinomial(probs, num_samples=1)
            # append to sequence
            idx = torch.cat((idx,idx_next), dim=1)
        return idx

model = MultiHeadAttentionLanguageModel()
m = model.to(device)

# optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
for iter in range(max_iters):
    # periodically evaluate the validation loss
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    # sample minibatch
    xb, yb = get_batch('train')
    # evaluate loss and take gradient step
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
# generate from model
idx = torch.zeros((1,1), dtype=torch.long, device=device)
new_seq = m.generate(idx, max_new_tokens=500)
print("predicted sequence:\n", new_seq)
print("... as a string of characters:\n", decode(new_seq[0].tolist()))


step 0: train loss 4.3103, val loss 4.3097
step 500: train loss 2.3986, val loss 2.3994
step 1000: train loss 2.2640, val loss 2.2658
step 1500: train loss 2.1660, val loss 2.1890
step 2000: train loss 2.1297, val loss 2.1660
step 2500: train loss 2.0793, val loss 2.1284
step 3000: train loss 2.0499, val loss 2.1242
step 3500: train loss 2.0420, val loss 2.1038
step 4000: train loss 2.0107, val loss 2.0927
step 4500: train loss 1.9912, val loss 2.0968
predicted sequence:
 tensor([[ 0,  0, 37, 27, 30, 35, 13, 30, 21, 34, 10,  0, 26, 53, 41, 46, 61, 47,
         50, 50,  1, 58, 46, 39, 58,  1, 44, 43, 50, 58,  1, 51, 39, 42, 47, 57,
         43, 42,  1, 40, 59, 40, 43, 56,  1, 39, 52, 42,  1, 31, 39, 45, 56, 59,
         39, 52, 57,  1, 51, 43, 39, 50, 50, 43, 42,  0, 25, 63,  1, 39, 56, 58,
         46, 47, 54,  1, 44, 53, 59, 50, 55,  1, 39, 50, 50,  1, 40, 59, 40, 50,
         43,  1, 42, 47, 50, 39, 46, 53, 39, 58, 43, 57,  1, 61, 47, 50, 50,  1,
         51, 63,  1, 44, 43, 39, 56, 

## Scaling up

- introduce `n_layer` for the number of blocks.
- to prevent overfitting, add dropout to output of the the fully connected layer, the multihead attention layer and the head's weights.
- scale up to the following parameters  
`batch_size = 64`  
`block_size = 256`  
`max_iters = 5000`  
`learning_rate = 3e-4`  
`n_embd = 384`  
`n_layer = 6`  
`n_heads = 6`  
`dropout = 0.2`

In [3]:
# %load multi_head_attention_v5.py
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 32
block_size = 8
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384 # 384/6 = 64, so head_size = 64
n_layer = 6 # number of blocks stacked sequentially
n_head = 6 # number of heads per block
dropout = 0.2

# seeding (for reproducibility)
torch.manual_seed(1337)

# open text file
filename = 'tinyshakespeare.txt' # 'tinymoliere.txt'
with open(filename, 'r', encoding='utf-8') as f:
    text = f.read()

# tokens
chars = sorted(list(set(text)))
vocab_size = len(chars)

# encode and decode strings
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encode a string as a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decode a list of integers as a string

# split dataset
data = torch.tensor(encode(text), dtype=torch.long)
n = int(.9*len(data))
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x,y

# estimate training and validation losses (for monitoring purposes)
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# single attention head
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size,block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self,x):
        B,T,C = x.shape
        k = self.key(x)   # B,T,C warning, here C stands for head_size
        q = self.query(x) # B,T,C
        # compute affinities and attention scores
        wei = q @ k.transpose(-2,-1) * C**-.5 # (B,T,T)
        wei = wei.masked_fill(self.tril[:T,:T]==0, float('-inf')) # (B,T,T)
        wei = F.softmax(wei, dim=-1) # (B,T,T)
        wei = self.dropout(wei)
        # aggregate values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B,T,C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self,x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    """ ReLU layer """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication then computation """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x)) # skip connection
        x = x + self.ffwd(self.ln2(x)) # skip connection
        return x

# Single attention head language model
class MultiHeadAttentionLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        # The logits for the next token are simply stored in a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T  = idx.shape
        # idx and targets are both (B,T)-shaped tensors of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C) 
        # warning here, T might grow larger than block_size if we generate more tokens than block_size, which will cause an index error in the previous line, but we can ignore that for now.
        x = tok_emb + pos_emb # the sum broadcasts the (T,C) tensor across batches
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        # at this point, logits are a (B,T,C)-shaped tensor
        # Pytorch expects a (n_data,n_classes) tensor to compute a CE loss
        # so we reshape the logits if we need to compute a loss
        # otherwise, keep the logits in the shape they are
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is a (B,T) array of indices
        for _ in range(max_new_tokens):
            # get prediction
            idx_cond = idx[:,-block_size:] # crop idx to the last block_size tokens
            logits, loss = self(idx_cond)
            logits = logits[:,-1,:]
            # apply softmax
            probs = F.softmax(logits, dim=-1)
            # sample next token
            idx_next = torch.multinomial(probs, num_samples=1)
            # append to sequence
            idx = torch.cat((idx,idx_next), dim=1)
        return idx

model = MultiHeadAttentionLanguageModel()
m = model.to(device)

# optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
for iter in range(max_iters):
    # periodically evaluate the validation loss
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    # sample minibatch
    xb, yb = get_batch('train')
    # evaluate loss and take gradient step
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
# generate from model
idx = torch.zeros((1,1), dtype=torch.long, device=device)
new_seq = m.generate(idx, max_new_tokens=500)
print("predicted sequence:\n", new_seq)
print("... as a string of characters:\n", decode(new_seq[0].tolist()))


step 0: train loss 4.3633, val loss 4.3610
step 500: train loss 2.1614, val loss 2.2141
step 1000: train loss 2.0201, val loss 2.1000
step 1500: train loss 1.9555, val loss 2.0541
step 2000: train loss 1.9033, val loss 2.0171
step 2500: train loss 1.8820, val loss 2.0059
step 3000: train loss 1.8505, val loss 1.9803
step 3500: train loss 1.8476, val loss 1.9719
step 4000: train loss 1.8169, val loss 1.9387
step 4500: train loss 1.8040, val loss 1.9452
predicted sequence:
 tensor([[ 0, 14, 39, 50, 43,  1, 46, 43, 56,  1, 47, 52,  1, 57, 53,  1, 57, 59,
         47, 41, 43,  1, 47, 58,  6,  1, 57, 46, 43,  1, 54, 56, 43, 45, 59, 43,
          0, 30, 53, 51, 43, 53,  1, 46, 47, 43,  1, 39, 57,  1, 40, 43,  1, 40,
         53, 43,  1, 58, 39, 47, 52, 10,  1, 46, 47, 54,  8,  0,  0, 20, 17, 26,
         30, 37,  1, 25, 27, 26, 32, 13, 10,  0, 14, 59, 57, 40, 39, 49, 43, 57,
          1, 47, 52,  1, 58, 46, 47, 52, 49, 53, 52,  1, 39, 52, 42,  1, 59, 54,
          1, 42, 53, 59, 45, 46, 58, 

## Extension to machine translation and cross-attention

This is a *decoder-only* Transformer. It does *completion* of a sequence, by doing character-by-character prediction, so that the output looks like the text given as training examples.  

The original *Attention is all you need* paper does *machine translation* from one language to another. This requires cross-attention (between tokens of the two languages) rather than self-attention only.  
The decoding part is the same as what we've done so far: we generate tokens in the target language using several piled-up blocks of self attention that feed on the tokens that have already been generated (in the target language). But we also want to condition the next-token prediction on the sequence of tokens of the sentence to translate (from the input language).  
Hence the "two-column" drawing in the paper. Queries come from the tokens in the target language, while keys and values come from the tokens in the input language. Queries have access to all *past* output token embeddings, and all the tokens in the input sentence.

## TODO next / comparison with the original repo's model.py

Bugs (these change behaviour)

1. The attention scale uses the wrong dimension (multi_head_attention_v5.py:79-83). C comes from x.shape, so it is n_embd = 384, not head_size = 64 as the comment on line 80 says. The scores are therefore scaled by 1/√384 instead of 1/√64, which makes attention about 2.4× too soft. nanoGPT uses 1.0 / math.sqrt(k.size(-1)). Fix: wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5.
2. Text is generated with dropout still on (:205-206). estimate_loss() puts the model back into train mode, and nothing calls model.eval() before generate, so 20% dropout is active while sampling. generate also runs without @torch.no_grad(), so it records gradient history for nothing. nanoGPT decorates generate with @torch.no_grad(), and its sample.py calls model.eval().

Attention differences (same maths, different implementation)

3. Separate heads vs one fused projection. You build n_head separate Head modules, each with its own q/k/v Linear, then join the outputs. nanoGPT uses one c_attn = Linear(n_embd, 3*n_embd), splits the result into q/k/v, and reshapes to (B, nh, T, hs). The results are the same but nanoGPT's version is much faster.
4. Flash attention. nanoGPT calls F.scaled_dot_product_attention(..., is_causal=True) when it is available. It only falls back to the manual tril mask without it.
5. Biases. Your q/k/v layers have bias=False and proj always has a bias. In nanoGPT, every Linear and LayerNorm follows one config.bias flag. It defaults to True, so its q/k/v layers do have biases.

Rest of the model

6. Activation: your FeedForward uses ReLU; nanoGPT's MLP uses GELU.
7. LayerNorm: you use nn.LayerNorm; nanoGPT uses its own LayerNorm class so the bias can be switched off.
8. Embedding dropout: nanoGPT applies dropout to tok_emb + pos_emb; yours has none.
9. Weight tying: nanoGPT shares the token embedding weights with lm_head; yours keeps them separate.
10. Initialisation: nanoGPT sets weights to N(0, 0.02), biases to zero, and scales the residual projections (c_proj) by 0.02/√(2·n_layer). You use PyTorch's defaults.
11. forward:
    - nanoGPT checks that T <= block_size; your comment on line 153 mentions this but there is no check.
    - nanoGPT builds positions on idx.device; you use the global device.
    - During inference nanoGPT computes logits only for the last position.
    - nanoGPT's loss uses ignore_index=-1.
12. generate: yours has no temperature or top_k options.

Training setup (these live in nanoGPT's train.py and config files, not model.py)

14. Optimizer:
    - You use Adam with one learning rate for everything.
    - nanoGPT uses AdamW with betas=(0.9, 0.95). Weight decay applies only to tensors with 2+ dimensions, not to biases or LayerNorm.
    - nanoGPT adds learning-rate warmup and cosine decay, gradient clipping at 1.0, mixed precision, and gradient accumulation.
15. Hyperparameters:
    - Context length: block_size = 8 is very short for a 6-layer, 384-wide model. nanoGPT's shakespeare_char config uses block_size=256.
    - Other settings: that config also uses batch_size=64 and learning_rate=1e-3.
    - Vocabulary: both are character-level here. nanoGPT's default GPT-2 setup instead uses BPE tokens with the vocabulary padded to 50304.

Items 1–3 are worth fixing first. I haven't changed the file; I can apply those fixes if you want.